# ROCLING 2026 DSA — E18 / E19 / E20 / E21（Arousal PCC 主攻）

第一波實驗（`dsa_nift_next_experiments_plan.md`）：**E4 已確認可復現，直接從 E18 開始。**

| 實驗 | 內容 | 指令關鍵 |
|---|---|---|
| E18 | L1++ intensity features（10→31 維） | `--lex_mode l1_intensity` |
| E19 | Source-aware Arousal loss（CVAS/CVAT arousal 降權） | `--source_aware` |
| E20 | train_aug 400 篇 ranking-only（不進 SmoothL1） | `--rank_aug data/train_aug.csv` |
| E21 | V/A dimension-specific attention pooling | `--pooling mean_cls_dim_attention` |

**對照基準 = 實驗 4（官方分數）**：V_MAE 0.600 / V_PCC 0.880 / A_MAE 0.882 / A_PCC 0.426

注意事項：
- **嚴格 batch 32 / lr 2e-5 / 4 epochs**，不要因為 A100 就調大 batch，否則與實驗 4 不可比。
- dev（DSA-MST 253 筆）**不是可靠代理**（尤其 E20 與合成資料同風格）→ 最終以官方提交分數為準。
- VS Code + Colab 擴充模式：產出在遠端 VM，**斷線即失**，跑完務必執行最後的打包 cell。

In [1]:
# 1) 安裝套件 + 確認 GPU
!pip -q install "transformers>=4.40" jieba scikit-learn scipy

import torch, subprocess
print('=' * 60)
if not torch.cuda.is_available():
    print('⚠️  沒有 GPU！請右上角 Select Kernel → Colab → 選 premium GPU runtime')
else:
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'✅ GPU：{name}（{vram:.1f} GB）')
print('=' * 60)
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

✅ GPU：NVIDIA A100-SXM4-40GB（39.5 GB）
Tue Jul 14 12:39:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             47W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------

In [2]:
# 2) git clone / pull 程式到 Colab runtime（token 用 getpass 輸入，不要寫進 notebook！）
import os, subprocess
from getpass import getpass

BRANCH = 'feat/ensemble-teacher-student-experiments'   # ← 程式所在 branch
REPO_PATH = '/content/repo' if os.path.exists('/content') else 'repo'

if not os.path.exists(REPO_PATH):
    token = getpass('GitHub token（public repo 直接按 Enter）: ').strip()
    prefix = f'{token}@' if token else ''
    !git clone -q -b {BRANCH} https://{prefix}github.com/chen0427ok/DSA-NIFT.git {REPO_PATH}
os.chdir(REPO_PATH)

# repo 已存在（舊 clone）時：切到正確 branch 並拉最新 commit
!git checkout -q {BRANCH}
r = subprocess.run(['git', 'pull', 'origin', BRANCH], capture_output=True, text=True)
print((r.stdout + r.stderr).strip())
if r.returncode != 0:
    # 舊 clone 的 remote URL 內嵌 token 可能已失效 → 用新 token 更新 remote 再拉一次
    token = getpass('git pull 失敗，輸入新的 GitHub token: ').strip()
    !git remote set-url origin https://{token}@github.com/chen0427ok/DSA-NIFT.git
    !git pull origin {BRANCH}

os.makedirs('outputs/preds', exist_ok=True)
print('工作目錄:', os.getcwd(), '| HEAD:',
      subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())
assert os.path.exists('train_v2.py'), '沒拉到程式，檢查 branch / token'
assert os.path.exists('lexicon_intensity.py'), '缺 lexicon_intensity.py → git pull 沒成功？'
print('✅ 程式就緒')
if os.path.exists('data/train_aug.csv'):
    print('✅ data/train_aug.csv 就緒（E20 可跑）')
else:
    print('⚠️ 缺 data/train_aug.csv（被 gitignore）→ E20 需要它！兩種帶上來的方式：')
    print('   A. 本機：git add -f data/train_aug.csv && git commit && git push，VM 上 git pull')
    print('   B. VS Code 檔案總管直接把本機 train_aug.csv 拖進 repo/data/')

Already up to date.
From https://github.com/chen0427ok/DSA-NIFT
 * branch            feat/ensemble-teacher-student-experiments -> FETCH_HEAD
工作目錄: /content/repo | HEAD: b0a9146 E18 / robertaL_s42 官方結果：四指標全輸，E18 消融揭露 E19 增益全來自 source-aware
✅ 程式就緒
✅ data/train_aug.csv 就緒（E20 可跑）


In [7]:
# 3) sanity check：L1++ 特徵抽取是否合理（31 維、cue 詞有命中）
!python lexicon_intensity.py

/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
Building prefix dict from the default dictionary ...
Dumping model to file cache /tmp/jieba.cache
Loading model cost 0.657 seconds.
Prefix dict has been built successfully.
詞典大小: 7761；特徵維度: 31

「我聽到那一刻簡直抓狂，胸口悶得喘不過氣！我躲在廁所裡發抖，氣到全身發冷。」
  coverage=0.13  count=0.15  v_mean=0.23  v_max=0.30  v_min=0.12
  v_std=0.07  a_mean=0.66  a_max=0.96  a_min=0.38  a_std=0.24
  exclam_density=0.93  question_density=0.00  ellipsis_density=0.00  degree_adverb=0.35  body_reaction=1.00
  sleep_disturb=0.00  anxi

## E18 — L1++ intensity features
L1 十維之外加 15 維 arousal intensity（標點密度、程度副詞、身體反應、睡眠、焦慮、壓力事件、
低喚醒詞、疊字、句長節奏、否定/轉折）+ 6 維新住民 domain cues。**不動資料分布**，
繞開 E13 的「校準↔排序」trade-off。期望：A_PCC ↑、A_MAE / Valence 不變差。

In [8]:
# 4) E18
!python train_v2.py --lex_mode l1_intensity \
    --run_name e18_l1_intensity --epochs 4 --batch_size 32 --lr 2e-5

run=e18_l1_intensity | device=cuda | model=hfl/chinese-macbert-base | lex=l1_intensity | aw=1.0 pcc_w=0.0 | src_aware=False | rank_aug=None | pooling=mean
config.json: 100% 659/659 [00:00<00:00, 4.08MB/s]
tokenizer_config.json: 100% 19.0/19.0 [00:00<00:00, 131kB/s]
vocab.txt: 100% 110k/110k [00:00<00:00, 86.4MB/s]
tokenizer.json: 100% 269k/269k [00:00<00:00, 112MB/s]
added_tokens.json: 100% 2.00/2.00 [00:00<00:00, 15.5kB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 806kB/s]
train=9435 dev=253 val=200
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.849 seconds.
Prefix dict has been built successfully.
lexicon dim = 31
pytorch_model.bin: 100% 412M/412M [00:04<00:00, 101MB/s] 
Loading weights: 100% 199/199 [00:00<00:00, 23766.81it/s]
[transformers] BertModel LOAD REPORT from: hfl/chinese-macbert-base
Key                                        | Status     |  | 
-------------------------------------------+-------

## E19 — Source-aware Arousal loss
train.csv 四個來源（granularity 欄）對 arousal 的 domain shift 不同 → V/A loss 分來源加權：
CVAS `1:0.25`、CVAT `1:0.5`、DSA-MST `1:1`、edu2021 `1:0.75`（V 全部 1.0）。
可用 `--source_weights "sentence=1:0.4,text=1:0.6"` 覆寫調權。

In [9]:
# 5) E19（疊在 L1++ 上）
!python train_v2.py --lex_mode l1_intensity --source_aware \
    --run_name e19_source_aware --epochs 4 --batch_size 32 --lr 2e-5

run=e19_source_aware | device=cuda | model=hfl/chinese-macbert-base | lex=l1_intensity | aw=1.0 pcc_w=0.0 | src_aware=True | rank_aug=None | pooling=mean
train=9435 dev=253 val=200
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.850 seconds.
Prefix dict has been built successfully.
lexicon dim = 31
source weights (V:A): {'sentence': (1.0, 0.25), 'text': (1.0, 0.5), 'reflection': (1.0, 1.0), 'edu2021': (1.0, 0.75), 'augment': (0.25, 0.0)}
Loading weights: 100% 199/199 [00:00<00:00, 19933.29it/s]
[transformers] BertModel LOAD REPORT from: hfl/chinese-macbert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTE

## E20 — Synthetic ranking-only augmentation（本波最有機會超越實驗 4）
`train_aug.csv` 400 篇**不進 SmoothL1**，只在每個 step 抽 16 篇建 batch 內 pairwise hinge：
gold 差 ≥1.5 分的 pair 要求預測同向拉開 margin 1 分。保留合成資料的排序訊號（實驗 9 的 A_PCC 0.46 來源），
不吃它不可靠的絕對標籤（A_MAE 爆掉的來源）。
期望：A_PCC → 0.46、A_MAE 守住 0.88–0.90。

In [10]:
# 6) E20（疊在 E19 上）
!python train_v2.py --lex_mode l1_intensity --source_aware \
    --rank_aug data/train_aug.csv --rank_lambda_a 0.1 --rank_lambda_v 0.05 \
    --run_name e20_rank_aug --epochs 4 --batch_size 32 --lr 2e-5

run=e20_rank_aug | device=cuda | model=hfl/chinese-macbert-base | lex=l1_intensity | aw=1.0 pcc_w=0.0 | src_aware=True | rank_aug=data/train_aug.csv | pooling=mean
train=9435 dev=253 val=200
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.857 seconds.
Prefix dict has been built successfully.
lexicon dim = 31
source weights (V:A): {'sentence': (1.0, 0.25), 'text': (1.0, 0.5), 'reflection': (1.0, 1.0), 'edu2021': (1.0, 0.75), 'augment': (0.25, 0.0)}
rank_aug: 400 rows (data/train_aug.csv) | λ_A=0.1 λ_V=0.05 margin=0.125 min_gap=0.1875
Loading weights: 100% 199/199 [00:00<00:00, 23480.65it/s]
[transformers] BertModel LOAD REPORT from: hfl/chinese-macbert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.tra

## E21 — Dimension-specific attention + multi-pooling fusion
mean pooling 會稀釋只出現在少數 token 的高喚醒 cue（心跳、睡不著、崩潰…）→
V/A 各自學一個 attention pooling，與 mean、CLS 拼接後分頭回歸（head dropout 0.2 抑制 overfit）。
先跑不含 rank 版；**若 E20 dev/official 有效**再跑第二個指令（E21+rank）。

In [11]:
# 7) E21a：dim attention（無 rank）
!python train_v2.py --pooling mean_cls_dim_attention --lex_mode l1_intensity --source_aware \
    --run_name e21_dim_attention --epochs 4 --batch_size 32 --lr 2e-5

run=e21_dim_attention | device=cuda | model=hfl/chinese-macbert-base | lex=l1_intensity | aw=1.0 pcc_w=0.0 | src_aware=True | rank_aug=None | pooling=mean_cls_dim_attention
train=9435 dev=253 val=200
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.849 seconds.
Prefix dict has been built successfully.
lexicon dim = 31
source weights (V:A): {'sentence': (1.0, 0.25), 'text': (1.0, 0.5), 'reflection': (1.0, 1.0), 'edu2021': (1.0, 0.75), 'augment': (0.25, 0.0)}
Loading weights: 100% 199/199 [00:00<00:00, 32855.71it/s]
[transformers] BertModel LOAD REPORT from: hfl/chinese-macbert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight        

In [12]:
# 8) E21b：dim attention + ranking（E20 有效才跑）
!python train_v2.py --pooling mean_cls_dim_attention --lex_mode l1_intensity --source_aware \
    --rank_aug data/train_aug.csv --rank_lambda_a 0.1 --rank_lambda_v 0.05 \
    --run_name e21_dim_attention_rank_aug --epochs 4 --batch_size 32 --lr 2e-5

run=e21_dim_attention_rank_aug | device=cuda | model=hfl/chinese-macbert-base | lex=l1_intensity | aw=1.0 pcc_w=0.0 | src_aware=True | rank_aug=data/train_aug.csv | pooling=mean_cls_dim_attention
train=9435 dev=253 val=200
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.854 seconds.
Prefix dict has been built successfully.
lexicon dim = 31
source weights (V:A): {'sentence': (1.0, 0.25), 'text': (1.0, 0.5), 'reflection': (1.0, 1.0), 'edu2021': (1.0, 0.75), 'augment': (0.25, 0.0)}
rank_aug: 400 rows (data/train_aug.csv) | λ_A=0.1 λ_V=0.05 margin=0.125 min_gap=0.1875
Loading weights: 100% 199/199 [00:00<00:00, 33951.61it/s]
[transformers] BertModel LOAD REPORT from: hfl/chinese-macbert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEX

## 結果彙總
dev 指標對照 + **official 200 篇 val 預測分布監控**（plan §10：注意 arousal std / range 是否被 ranking
過度放大，或被 source-aware 壓縮）。實驗 4 的 val Arousal pred std ≈ 0.755 可當參考。

In [13]:
# 9) 彙總所有 run 的 dev 指標 + val 分布
import pandas as pd, numpy as np, glob, os

rows = []
for p in sorted(glob.glob('outputs/preds/*_dev.csv')):
    df = pd.read_csv(p)
    r = {'run': os.path.basename(p)[:-8]}
    for d in ['valence', 'arousal']:
        r[f'{d[0].upper()}_MAE'] = np.abs(df[f'{d}_pred'] - df[f'{d}_true']).mean()
        r[f'{d[0].upper()}_PCC'] = np.corrcoef(df[f'{d}_pred'], df[f'{d}_true'])[0, 1]
    rows.append(r)
print('=== dev (253) — 僅供參考，dev 對增強實驗會失真 ===')
print(pd.DataFrame(rows).round(4).to_string(index=False))

print('\n=== official 200 val 預測分布 ===')
for p in sorted(glob.glob('outputs/*_submission.csv')):
    df = pd.read_csv(p)
    print(f"{os.path.basename(p):44s} V {df.Valence.mean():.2f}±{df.Valence.std():.2f}  "
          f"A {df.Arousal.mean():.2f}±{df.Arousal.std():.2f}  "
          f"A range [{df.Arousal.min():.2f}, {df.Arousal.max():.2f}]")

=== dev (253) — 僅供參考，dev 對增強實驗會失真 ===
                       run  V_MAE  V_PCC  A_MAE  A_PCC
              e10_seed_ens 0.4720 0.8229 0.8256 0.6157
               e12_enc_ens 0.4578 0.8291 0.8268 0.6153
              e12_enc_mean 0.4582 0.8289 0.8271 0.6152
          e18_l1_intensity 0.4789 0.8207 0.8311 0.6126
          e19_source_aware 0.4782 0.8206 0.8231 0.6180
              e20_rank_aug 0.4829 0.8080 0.8300 0.6107
         e21_dim_attention 0.4808 0.8168 0.8332 0.6167
e21_dim_attention_rank_aug 0.4894 0.8281 0.8307 0.6163
         macbert_pseudo_s1 0.4911 0.8162 0.8434 0.6020
         macbert_pseudo_s2 0.4838 0.8145 0.8336 0.6096
        macbert_pseudo_s42 0.4760 0.8199 0.8333 0.6039
                macbert_s1 0.4905 0.8129 0.8390 0.6154
                macbert_s2 0.4813 0.8113 0.8301 0.6085
                macbert_s3 0.4922 0.8127 0.8516 0.6042
               macbert_s42 0.4716 0.8189 0.8241 0.6146
                macbert_s4 0.4764 0.8176 0.8447 0.6038
              robertaL_s42 

In [14]:
# 10) 打包結果（斷線即失！跑完務必執行）
import shutil, os
zip_path = shutil.make_archive('e18_e21_results', 'zip', 'outputs')
print('已打包 ->', os.path.abspath(zip_path), f'({os.path.getsize(zip_path)/1024**3:.2f} GB)')
try:
    from google.colab import files
    files.download('e18_e21_results.zip')
except Exception:
    print('VS Code 模式無彈窗下載：')
    print('  方式 A：左側檔案總管找 repo/e18_e21_results.zip 右鍵 Download')
    print('  方式 B：下一個 cell 複製到 Google Drive')

已打包 -> /content/repo/e18_e21_results.zip (1.78 GB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# 11) （可選）複製到 Google Drive
import shutil, os
from google.colab import drive
drive.mount('/content/drive')
dst = shutil.copy('/content/repo/e18_e21_results.zip', '/content/drive/MyDrive/')
print(f'✅ 已複製到 Drive：{dst}')

Mounted at /content/drive
✅ 已複製到 Drive：/content/drive/MyDrive/e18_e21_results.zip


In [16]:
# 12) （可選）把 submission / preds 小檔 push 回 branch（權重不 push）
import os
os.chdir('/content/repo')
!git config user.email "chenbrian930427@gmail.com"
!git config user.name "chen0427ok"
!git add -f outputs/e1*_submission.csv outputs/e2*_submission.csv outputs/preds/*.csv
!git status --short
!git commit -m "E18-E21 結果：submission 與 dev/val 預測檔"
!git push origin feat/ensemble-teacher-student-experiments

A  outputs/e18_l1_intensity_submission.csv
A  outputs/e19_source_aware_submission.csv
A  outputs/e20_rank_aug_submission.csv
A  outputs/e21_dim_attention_rank_aug_submission.csv
A  outputs/e21_dim_attention_submission.csv
A  outputs/preds/e18_l1_intensity_dev.csv
A  outputs/preds/e18_l1_intensity_val.csv
A  outputs/preds/e19_source_aware_dev.csv
A  outputs/preds/e19_source_aware_val.csv
A  outputs/preds/e20_rank_aug_dev.csv
A  outputs/preds/e20_rank_aug_val.csv
A  outputs/preds/e21_dim_attention_dev.csv
A  outputs/preds/e21_dim_attention_rank_aug_dev.csv
A  outputs/preds/e21_dim_attention_rank_aug_val.csv
A  outputs/preds/e21_dim_attention_val.csv
?? repo/
[feat/ensemble-teacher-student-experiments e114dec] E18-E21 結果：submission 與 dev/val 預測檔
 15 files changed, 3280 insertions(+)
 create mode 100644 outputs/e18_l1_intensity_submission.csv
 create mode 100644 outputs/e19_source_aware_submission.csv
 create mode 100644 outputs/e20_rank_aug_submission.csv
 create mode 100644 outputs/e21_d

In [3]:
!python train_v2.py --lex_mode l1 --source_aware \
      --run_name e22_l1_source_aware --epochs 4 --batch_size 32 --lr 2e-5

/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
run=e22_l1_source_aware | device=cuda | model=hfl/chinese-macbert-base | lex=l1 | aw=1.0 pcc_w=0.0 | src_aware=True | rank_aug=None | pooling=mean
config.json: 100% 659/659 [00:00<00:00, 4.31MB/s]
tokenizer_config.json: 100% 19.0/19.0 [00:00<00:00, 141kB/s]
vocab.txt: 100% 110k/110k [00:00<00:00, 86.9MB/s]
tokenizer.json: 100% 269k/269k [00:00<00:00, 121MB/s]
added_tokens.json: 100% 2.00/2.00 [00:00<00:00, 15.0kB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 875kB/s]
train=9435 de